# Movie Recommender System with Matrix Factorization (SVD)

**Goal:** build a collaborative filtering recommender using Singular Value Decomposition (SVD), and measure whether it actually beats a simple popularity baseline.

**Approach:** SVD-based collaborative filtering assumes user ratings can be explained by a small number of hidden factors (think: genre taste, mood, era preference) that aren't directly labeled in the data. By decomposing the user-movie rating matrix into `U`, `sigma`, and `Vt`, we can reconstruct a dense prediction matrix that fills in ratings for movies a user hasn't seen yet, based on patterns learned from everyone else.

**Dataset:** [MovieLens Latest Small](https://grouplens.org/datasets/movielens/latest/) (~100K ratings, ~600 users, ~9K movies). Unzip `ml-latest-small.zip` so the folder sits next to this notebook.

**Evaluation:** for each user, 20% of ratings are hidden before training. We measure:
- **RMSE** — how close predicted star ratings are to actual hidden ratings
- **Precision@10 / Recall@10** — of the top 10 movies recommended, how many did the user actually like (4+ stars) in the hidden set, compared against a popularity baseline

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse.linalg import svds

np.random.seed(42)

## 1. Load the Data

In [ ]:
DATA_DIR = "ml-latest-small"

movies = pd.read_csv(f"{DATA_DIR}/movies.csv")
ratings = pd.read_csv(f"{DATA_DIR}/ratings.csv")

print("movies:", movies.shape)
print("ratings:", ratings.shape)
ratings.head()

## 2. Explore the Data

SVD needs a dense-enough matrix per user to find meaningful patterns, so it helps to know how sparse the data is up front.

In [ ]:
n_users = ratings["userId"].nunique()
n_movies = ratings["movieId"].nunique()
sparsity = 1 - len(ratings) / (n_users * n_movies)
print(f"{n_users} users, {n_movies} movies, {len(ratings)} ratings")
print(f"Sparsity: {sparsity:.2%} of the user-movie matrix is empty")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ratings["rating"].plot(kind="hist", bins=10, ax=axes[0], title="Rating Distribution")
ratings.groupby("userId").size().plot(kind="hist", bins=30, ax=axes[1], title="Ratings per User")
plt.tight_layout()
plt.show()

**Your turn:** write 2-3 sentences on what these plots show (rating skew, how many ratings a typical user has) and reference the sparsity number.

## 3. Filter Rare Movies and Split Train/Test

Movies with very few ratings add noise without adding signal, so they're dropped before building the matrix. Then, for each user, 20% of their ratings are held out as a hidden test set — splitting *per user* guarantees every user appears in both train and test.

In [ ]:
MIN_RATINGS = 10
movie_counts = ratings["movieId"].value_counts()
keep_ids = movie_counts[movie_counts >= MIN_RATINGS].index
ratings = ratings[ratings["movieId"].isin(keep_ids)].copy()
movies = movies[movies["movieId"].isin(keep_ids)].reset_index(drop=True)

print(f"After filtering: {ratings['userId'].nunique()} users, {movies.shape[0]} movies, {len(ratings)} ratings")

In [ ]:
ratings = ratings.sample(frac=1, random_state=42).reset_index(drop=True)
ratings["pos"] = ratings.groupby("userId").cumcount()
ratings["n_user"] = ratings.groupby("userId")["rating"].transform("size")

is_test = ratings["pos"] < 0.2 * ratings["n_user"]
train = ratings[~is_test].copy()
test = ratings[is_test].copy()

print(f"Train: {len(train)} ratings | Test: {len(test)} ratings")

## 4. Build the User-Item Matrix (Training Data Only)

This is the core structure for collaborative filtering: rows are users, columns are movies, and each cell is that user's rating (or blank if unrated). It's built from **training ratings only** — the test ratings stay hidden until evaluation.

In [ ]:
user_item_matrix = train.pivot_table(index="userId", columns="movieId", values="rating")
print(user_item_matrix.shape)
user_item_matrix.head()

## 5. Mean-Center the Ratings

Some users rate everything a 4 or 5; others are harsher critics who rarely go above a 3. Subtracting each user's average rating removes this bias so SVD learns *relative* preferences rather than just who's generous with stars. Missing values are then filled with 0 (the centered "neutral" point) so SVD has a complete matrix to decompose.

In [ ]:
user_means = user_item_matrix.mean(axis=1)
centered_ratings = user_item_matrix.sub(user_means, axis=0)
centered_ratings = centered_ratings.fillna(0)
centered_ratings.head()

## 6. Decompose with SVD

`svds` factors the matrix into three pieces: `U` (user factors), `sigma` (the strength of each factor), and `Vt` (movie factors). `k=10` means we're compressing all rating patterns down to 10 hidden dimensions — a form of dimensionality reduction that also smooths out noise in the sparse original matrix.

In [ ]:
k = min(10, min(centered_ratings.shape) - 1)
U, sigma, Vt = svds(centered_ratings.to_numpy(), k=k)

# svds returns factors in ascending order of importance; flip to descending
order = np.argsort(sigma)[::-1]
sigma = sigma[order]
U = U[:, order]
Vt = Vt[order, :]

print("U:", U.shape, "| sigma:", sigma.shape, "| Vt:", Vt.shape)

## 7. Reconstruct Predicted Ratings

Multiplying the three factors back together reconstructs a **dense** matrix — no more missing values. Every user now has a predicted rating for every movie, including ones they've never seen. Adding back `user_means` reverses the centering from step 5, returning predictions to the original 0.5-5 star scale.

In [ ]:
sigma_matrix = np.diag(sigma)
centered_prediction = np.dot(np.dot(U, sigma_matrix), Vt)
centered_prediction = pd.DataFrame(
    centered_prediction, index=user_item_matrix.index, columns=user_item_matrix.columns
)

predicted_ratings = centered_prediction.add(user_means, axis=0)
predicted_ratings.head()

## 8. Evaluate: RMSE on Held-Out Ratings

For every rating in the hidden test set, compare the SVD model's prediction to the actual star rating the user gave. RMSE (root mean squared error) tells us, on average, how many stars off the predictions are. Test ratings for movies that never appeared in training can't be predicted, so they're skipped (a known limitation of this approach called the *cold-start problem*).

In [ ]:
test_eval = test[test["movieId"].isin(predicted_ratings.columns)]

preds = predicted_ratings.lookup(test_eval["userId"], test_eval["movieId"]) \
    if hasattr(predicted_ratings, "lookup") else \
    np.array([predicted_ratings.at[u, m] for u, m in zip(test_eval["userId"], test_eval["movieId"])])

actual = test_eval["rating"].values
rmse = np.sqrt(np.mean((preds - actual) ** 2))
print(f"Evaluated on {len(test_eval)}/{len(test)} test ratings (rest were cold-start movies)")
print(f"SVD RMSE: {rmse:.4f} stars")

## 9. Evaluate: Precision@10 / Recall@10 vs. a Popularity Baseline

RMSE alone doesn't tell us whether the *top 10 recommendations* are any good, and it doesn't say whether all this matrix factorization is even worth it over just recommending what's popular. This section builds both models and compares them head-to-head.

In [ ]:
liked_test = test_eval[test_eval["rating"] >= 4]
relevant = {
    uid: set(group["movieId"])
    for uid, group in liked_test.groupby("userId")
}
print(f"{len(relevant)} users have at least one liked movie in the evaluable test set")

In [ ]:
def evaluate(score_lookup_fn, k=10):
    """score_lookup_fn(user_id) -> pd.Series of scores indexed by movieId"""
    precisions, recalls = [], []
    for uid, rel in relevant.items():
        if uid not in user_item_matrix.index:
            continue
        seen = user_item_matrix.loc[uid].dropna().index
        scores = score_lookup_fn(uid).drop(index=seen, errors="ignore")
        top_k = set(scores.nlargest(k).index)
        hits = len(top_k & rel)
        precisions.append(hits / k)
        recalls.append(hits / len(rel))
    return np.mean(precisions), np.mean(recalls)

svd_p, svd_r = evaluate(lambda uid: predicted_ratings.loc[uid])
print(f"SVD CF      | Precision@10: {svd_p:.4f} | Recall@10: {svd_r:.4f}")

In [ ]:
movie_popularity = train[train["rating"] >= 4]["movieId"].value_counts()
movie_popularity = movie_popularity.reindex(user_item_matrix.columns, fill_value=0)

pop_p, pop_r = evaluate(lambda uid: movie_popularity)
print(f"Popularity  | Precision@10: {pop_p:.4f} | Recall@10: {pop_r:.4f}")

In [ ]:
results = pd.DataFrame({
    "Model": ["Popularity", "SVD Collaborative Filtering"],
    "Precision@10": [pop_p, svd_p],
    "Recall@10": [pop_r, svd_r],
}).set_index("Model")

display(results.round(4))
results.plot(kind="bar", figsize=(7, 4), rot=0, title="Model Comparison (higher is better)")
plt.ylabel("Score")
plt.tight_layout()
plt.show()

**Your turn:** write your conclusions here. Questions to answer:

- Did SVD beat the popularity baseline? By how much?
- What does the RMSE number mean in plain English (e.g. "predictions are typically within X stars")?
- SVD compresses everything into `k=10` hidden factors. What might it be missing that genre or tag data could capture?
- What would you try next? (Ideas: tune `k`, add a content-based model as a third comparison, build a simple hybrid.)

## 10. Get Recommendations for a User

In [ ]:
def collaborative_recommender(user_id, n=5):
    unseen_mask = user_item_matrix.loc[user_id].isna()
    predicted_for_unseen = predicted_ratings.loc[user_id, unseen_mask]
    top = predicted_for_unseen.nlargest(n)

    top = top.rename("predicted_rating").rename_axis("movieId").reset_index()
    result = pd.merge(top, movies, on="movieId", how="left")
    return result[["movieId", "title", "predicted_rating"]]

In [ ]:
# Change the user ID to any userId in user_item_matrix.index
collaborative_recommender(user_item_matrix.index[0], n=5)

## Summary and Next Steps

Fill this in once you've run everything on the real dataset: state the problem, your RMSE, your Precision@10/Recall@10 versus the popularity baseline, and what that comparison tells you about whether matrix factorization was worth the added complexity.

**Stretch goals:**
1. Tune `k` (try 5, 20, 50) and plot how RMSE and Precision@10 change.
2. Add the content-based genre model from your other project as a third point of comparison.
3. Build a simple hybrid that averages SVD and content-based scores.